ARC-AGI-2: object-DSL program induction (symbolic, no neural model)
Candidate: commit 9d446873 (frozen bundle sha256 9eedcc40...). Per task, a short program over
ARCGraph abstract objects is inferred from the training pairs only, then executed on test inputs.
Unresolved slots get a 1x1 placeholder, so any non-zero score comes from the inferred programs.

In [ ]:
import hashlib, json, subprocess, sys, zipfile
from pathlib import Path
BUNDLE_SHA = '9eedcc404d6015a2c84db58c63ec97c3e55f282faaed5fb9eb479afe0a6ab4fb'
bundles = [p for p in Path('/kaggle/input').rglob('candidate.bundle')]
assert len(bundles) == 1, bundles
payload = bundles[0].parent
assert hashlib.sha256(bundles[0].read_bytes()).hexdigest() == BUNDLE_SHA
cand = Path('/kaggle/working/cand')
if not cand.exists():
    with zipfile.ZipFile(bundles[0]) as z:
        z.extractall(cand)
deps = Path('/kaggle/working/deps')
try:
    import rdflib  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--target', str(deps),
                    str(payload / 'rdflib-7.6.0-py3-none-any.whl')], check=True)
challenges = [p for p in Path('/kaggle/input').rglob('arc-agi_test_challenges.json')]
assert len(challenges) == 1, challenges
print(payload, challenges[0])

In [ ]:
%%writefile /kaggle/working/predict.py
"""Competition driver for the frozen object-DSL candidate (commit 9d446873).

Symbolic only: every prediction is produced by a program inferred from the
task's own training pairs by experiments/object_dsl_recovery/search.py.
Unresolved slots receive a 1x1 placeholder that cannot plausibly be correct,
so any non-zero score is attributable to the DSL. Per-task failures (fragment
gaps, query gaps, timeouts, unexpected errors) are recorded and never abort
the batch.
"""
from __future__ import annotations

import argparse
import json
import signal
import sys
import time
from pathlib import Path

PLACEHOLDER = [[0]]


class TaskTimeout(Exception):
    pass


def _alarm(signum, frame):
    raise TaskTimeout()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--candidate", type=Path, required=True)
    parser.add_argument("--challenges", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--log", type=Path, required=True)
    parser.add_argument("--task-seconds", type=int, default=90)
    parser.add_argument("--total-seconds", type=int, default=6 * 3600)
    args = parser.parse_args()

    sys.path.insert(0, str(args.candidate))
    from experiments.object_dsl_recovery.search import infer, execute, Limits

    challenges = json.loads(args.challenges.read_text(encoding="utf-8"))
    submission, log = {}, []
    started = time.monotonic()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, _alarm)

    for key in sorted(challenges):
        task = challenges[key]
        slots = [{"attempt_1": PLACEHOLDER, "attempt_2": PLACEHOLDER} for _ in task["test"]]
        submission[key] = slots
        row = {"task": key, "status": "unattempted", "predicted": 0}
        if time.monotonic() - started > args.total_seconds:
            row["status"] = "global_budget_exhausted"
            log.append(row)
            continue
        t0 = time.monotonic()
        try:
            if use_alarm:
                signal.alarm(args.task_seconds)
            found = infer(task["train"], Limits(evaluations=256))
            program = found.get("program")
            row["status"] = found.get("status")
            if program is not None:
                for index, example in enumerate(task["test"]):
                    try:
                        grid = execute(example["input"], program)["grid"]
                        if grid and all(isinstance(r, list) and r for r in grid):
                            slots[index]["attempt_1"] = grid
                            row["predicted"] += 1
                    except TaskTimeout:
                        raise
                    except Exception as error:  # out-of-fragment test input
                        row.setdefault("test_errors", []).append(f"{index}:{type(error).__name__}:{error}")
        except TaskTimeout:
            row["status"] = "task_timeout"
        except Exception as error:
            row["status"] = f"error:{type(error).__name__}:{error}"[:300]
        finally:
            if use_alarm:
                signal.alarm(0)
        row["seconds"] = round(time.monotonic() - t0, 3)
        log.append(row)

    args.out.write_text(json.dumps(submission), encoding="utf-8")
    summary = {"tasks": len(challenges), "slots": sum(len(v) for v in submission.values()),
               "predicted_slots": sum(r["predicted"] for r in log),
               "seconds": round(time.monotonic() - started, 3), "rows": log}
    args.log.write_text(json.dumps(summary, indent=1), encoding="utf-8")
    print(json.dumps({k: v for k, v in summary.items() if k != "rows"}))


if __name__ == "__main__":
    main()


In [ ]:
# Parity gate: replicate the local score on the public 120-task / 172-slot ARC-2 evaluation set
# before producing the competition submission. Fails the notebook if predictions differ from the
# frozen local run (commit 9d446873): 2 correct / 170 unresolved, prediction digest below.
import hashlib, json, os
EVAL_SHA = 'e7c62a4bd211867c6b538f66b8013b81f299663c82ca062f49a52bf439d6e4e8'
EXPECTED_DIGEST = '384a717bcbfb2ded048019299885791513fcf67360e5b437350db91a8c195468'
evals = [p for p in Path('/kaggle/input').rglob('arc-agi_evaluation_challenges.json')
         if hashlib.sha256(p.read_bytes()).hexdigest() == EVAL_SHA]
assert evals, 'identical public evaluation file not found; parity cannot be verified'
sols = list(evals[0].parent.glob('arc-agi_evaluation_solutions.json'))
env = dict(os.environ, PYTHONHASHSEED='0', OMP_NUM_THREADS='1', MPLBACKEND='Agg',
           PYTHONPATH=os.pathsep.join([str(deps), os.environ.get('PYTHONPATH', '')]))
subprocess.run([sys.executable, '/kaggle/working/predict.py', '--candidate', str(cand),
                '--challenges', str(evals[0]), '--out', '/kaggle/working/parity_eval.json',
                '--log', '/kaggle/working/parity_eval_log.json', '--task-seconds', '90'], check=True, env=env)
ev = json.load(open('/kaggle/working/parity_eval.json'))
preds = {k: [s['attempt_1'] for s in v] for k, v in sorted(ev.items())}
digest = hashlib.sha256(json.dumps(preds, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
correct = None
if sols:
    ans = json.load(open(sols[0]))
    correct = sum(s == t for k in ans for s, t in zip(preds[k], ans[k]))
report = {'eval_file': str(evals[0]), 'prediction_digest': digest, 'digest_match': digest == EXPECTED_DIGEST,
          'correct_of_172': correct}
json.dump(report, open('/kaggle/working/parity_report.json', 'w'), indent=1)
print(report)
assert digest == EXPECTED_DIGEST, 'PARITY FAILED: predictions differ from the frozen local run'
assert correct in (None, 2), f'PARITY FAILED: expected 2 correct, got {correct}'

In [ ]:
import os
env = dict(os.environ, PYTHONHASHSEED='0', OMP_NUM_THREADS='1', MPLBACKEND='Agg',
           PYTHONPATH=os.pathsep.join([str(deps), os.environ.get('PYTHONPATH', '')]))
subprocess.run([sys.executable, '/kaggle/working/predict.py', '--candidate', str(cand),
                '--challenges', str(challenges[0]), '--out', '/kaggle/working/submission.json',
                '--log', '/kaggle/working/predict_log.json', '--task-seconds', '90',
                '--total-seconds', str(8 * 3600)], check=True, env=env)
sub = json.load(open('/kaggle/working/submission.json'))
src = json.load(open(challenges[0]))
assert set(sub) == set(src) and all(len(sub[k]) == len(src[k]['test']) for k in src)
print(len(sub), 'tasks written')